# Diagnosing model issues using hydrographs and storage timeseries

## Setup

In [ ]:
%matplotlib widget

import os
import sys
import pandas as pd
import spotpy as sp

# Ensure the project directory is in the Python path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)
import src.py.raven_py as rv
from src.py.proj_conf import *
from src.py.setup_spotpy import *

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
raven_name = 'Cypress_New'

cal_data_path = DATA_CAL + "../Results/fast_rdrs_toffset_20251002_21"

In [ ]:
def combine_solutions(sol_dataset, fixed_vals = {}, hi_lim_vals = {}, low_lim_vals = {}):
    # Combine solutions from a spotpy dataset into a single parameter set
    # Get the parameter names from the dataset
    param_names = sol_dataset.columns[1:-2]  # Exclude the first (index) and last (objective) columns

    # Create a dictionary to hold combined parameter values
    combined_params = {}

    for hlim in hi_lim_vals.keys():
        sol_dataset = sol_dataset[sol_dataset[hlim] < hi_lim_vals[hlim]]
    
    for llim in low_lim_vals.keys():
        sol_dataset = sol_dataset[sol_dataset[llim] > low_lim_vals[llim]]

    for param in param_names:
        if param in fixed_vals:
            combined_params[param] = fixed_vals[param]
        else:
            combined_params[param] = np.median(sol_dataset[param])

    # for hlim in hi_lim_vals.keys():
    #     combined_params[hlim] = np.median(sol_dataset[hlim][sol_dataset[hlim] < hi_lim_vals[hlim]])

    # for llim in low_lim_vals.keys():
    #     combined_params[llim] = np.median(sol_dataset[llim][sol_dataset[llim] > low_lim_vals[llim]])



    return pd.Series(combined_params)

# Custom Raven Simulations

In [ ]:
# Load Raven input data

aspect_ratio_path = "../Data/Calibration/Cypress_New/input/aspect_ratios.csv"
aspect_ratios = pd.read_csv(aspect_ratio_path).set_index("SubId")["Aspect_Ratio"].to_dict()

HRU_areas_path = "../Data/Calibration/Cypress_New/input/Cypress_New_hru_areas.csv"
HRU_areas = pd.read_csv(HRU_areas_path).set_index("HRU")["AREA"].to_dict()

tp_tc_path = "../Data/Calibration/Cypress_New/input/Cypress_New_tp_tc.csv"
tps = pd.read_csv(tp_tc_path).set_index("SubId")["tp"].to_dict()
tcs = pd.read_csv(tp_tc_path).set_index("SubId")["tc"].to_dict()

deps_path = "../Data/Calibration/Cypress_New/input/CLRH_basins_depression_depths.shp"
deps_gdf = gpd.read_file(deps_path)
qrefs = dict(zip(deps_gdf["SubId"], deps_gdf["Q_Mean"]))
deps = dict(zip(deps_gdf["SubId"], deps_gdf["Deps_Depth"]))

In [ ]:
# Load results of the spotpy calibration
calibration_results = pd.read_csv(cal_data_path + ".csv")

# Option to filter results based on parameter limits
calibration_results = calibration_results[(calibration_results['parbasin_fact'] > 1) & (calibration_results['parseep_k'] > 0.15)]

# Select a result record to use as the solution
# solution = calibration_results[calibration_results['like1'] == max(calibration_results['like1'])][0]
solution = calibration_results.sort_values(by='like1', ascending=False).iloc[0]

# Or combine them

# calibration_results = calibration_results.sort_values(by='like1', ascending=False).head(100)
# solution = combine_solutions(calibration_results, low_lim_vals = {'parseep_k': 0.15})

# Clean the names from the spotpy output format
solution.index = solution.index.str.replace('par', '')
# Sanity check
solution

In [ ]:
# solution.dep_fact = 0.5
# solution.seep_k = 0.4
# solution.basin_fact = 1.5
# solution.bedslope_mult = 4.5


In [ ]:
# User inputs for diagnostic raven run
diagnostic_path =  "../Data/Diagnostics/Cypress_New/"

run_id = input('Enter run ID: ')

if not os.path.exists(f'{diagnostic_path}/output_{run_id}'):
    print(f"Warning: Output directory does not exist for run_id {run_id}.")

At this point, it may be necessary to check and manually edit the rvh.tpl file to mark "gauged" HRUs for data export

In [ ]:
# Update Raven input files with the selected parameters



rv.RavenFile(filename = raven_name, calib_dir = diagnostic_path, extension='rvh').rvh_update('no_split', True, True, solution, aspect_ratios, HRU_areas, tps, tcs, qrefs)
rv.RavenFile(filename = raven_name, calib_dir = diagnostic_path, extension='rvp').rvp_update('no_split', True, solution, deps)
rv.RavenFile(filename = 'channel_properties', calib_dir = diagnostic_path, extension='rvp').rvp_update('no_split', True, solution, deps)

Reminder: If using a new NetCDF file, the .rvt file should be updated manually. 

In [ ]:

rv.run_raven(raven_name, diagnostic_path, output_dir=f'output_{run_id}')

## Hydrographs & Evaluation Metrics

In [ ]:
# Load Hydrographs

test = rv.RavenHydrograph(diagnostic_path, _run_id=run_id)

sim = test.read_sim(subbasin_id = "sub310019543", start_yr=2003, end_yr=2021, spring_only = False, refresh=True).reset_index(drop=True)
# sim_full = self.hydrograph.read_sim(subbasin_id, start_yr=2003, end_yr=2021, spring_only = False, refresh=True).reset_index(drop=True) # From setup_spotpy.py
eval = test.read_obs(start_yr=2003, end_yr=2021, spring_only = False, refresh=True).reset_index(drop=True)
# obs_full = self.hydrograph.read_obs(start_yr=2003, end_yr=2021, spring_only = False, refresh=True).reset_index(drop=True) # From setup_spotpy.py
# runoff = test.read_runoff(start_yr=2003, end_yr=2021, spring_only = False, refresh=True).reset_index(drop=True)

In [ ]:
def clean_sim_eval(_sim, _eval):
    # Identify null values in the observations
    non_null = _eval.notnull()

    # Remove corresponding rows from the simulation and observation data
    valid_sim = _sim[non_null]
    valid_eval = _eval[non_null]

    # Reset the start date
    valid_start = _eval[_eval.notnull()].index[0]

    return {'sim':valid_sim, 'eval': valid_eval, 'start': valid_start}

hydro_clean_full = clean_sim_eval(sim, eval)

In [ ]:
spring_sim = test.read_sim(subbasin_id = "sub310019543", start_yr=2003, end_yr=2021, spring_only = True, refresh=True).reset_index(drop=True)
# sim = sim_hydrograph.read_sim(subbasin_id, start_yr=2003, end_yr=2021, spring_only = self.spring_eval, refresh=True).reset_index(drop=True)
spring_eval = test.read_obs(start_yr=2003, end_yr=2021, spring_only = True, refresh=True).reset_index(drop=True)
#  obs = self.hydrograph.read_obs(start_yr=2003, end_yr=2021, spring_only = self.spring_eval, refresh=True).reset_index(drop=True)

hydro_clean_spring = clean_sim_eval(spring_sim, spring_eval)

### Verify hydrograph diagnostics

In [ ]:

nse = 1- np.sum((hydro_clean_spring['sim'] - hydro_clean_spring['eval']) ** 2) / np.sum((hydro_clean_spring['eval'] - np.mean(hydro_clean_spring['eval'])) ** 2)
# nse = pd.Series(1- np.sum((valid_sim - valid_eval) ** 2) / np.sum((valid_eval - np.mean(valid_eval)) ** 2)) # From setup_spotpy.py
nse = np.squeeze(nse).item()
print("Nash-Sutcliffe Efficiency (NSE):", nse)

In [ ]:
# pbias = np.mean(sim-eval)/np.mean(eval) * 100
# pbias = pd.Series(np.sum(hydro_clean_full['sim'] - hydro_clean_full['eval'])/np.sum(hydro_clean_full['eval']) * 100)
# pbias = pd.Series(np.sum(sim_full - obs_full) / np.sum(obs_full)) # From setup_spotpy.py
pbias = np.sum(hydro_clean_spring['sim'] - hydro_clean_spring['eval'])/np.sum(hydro_clean_spring['eval']) * 100

print(f'Percent Bias (PBIAS): {pbias} %')

In [ ]:
np.squeeze(2*(nse-1)-np.sqrt(pbias ** 2)/100)
# weighted_obj = 2*(nse - 1) - np.sqrt(pbias**2) # from setup_spotpy.py

### Option to test the spotpy calibrator functions

In [ ]:
sim_input = pd.Series(solution.tolist(), index = param_names)
# sim_input = pd.Series(sim_df.iloc[0].tolist(), index = param_names)

In [ ]:
calibrator = spot_setup_rvn(raven_name, DATA_CAL, obj_fn= "nse_pbias", split_areas = "none", \
                            spring_eval= True, low_seep=False, routing=True, dep_scaling=True, runoff_scaling=True, 
                            mpi_cores=0)

calibrator.simulation(sim_input)

# Full Watershed Timeseries Plots

Reading Raven storage timeseries

In [ ]:
# DATA_CAL = "../Data/Calibration/cypress"

file_path = diagnostic_path + f"/output{test.run_id}/" + \
    "Cypress_New" \
    # "CypressRiverT_ERA5" \ 
file_path = file_path + "_WatershedStorage.csv"


In [ ]:
storage_ts = pd.read_csv(file_path)
storage_ts["timestamp"] = pd.to_datetime(storage_ts['date'] + " " + storage_ts['hour'].str[:8], format='%Y-%m-%d %H:%M:%S')
storage_ts = storage_ts[storage_ts['rainfall [mm/day]'] != '---']
storage_ts['rainfall [mm/day]'] = storage_ts['rainfall [mm/day]'].astype(float)
storage_ts.columns

In [ ]:
plt.figure()
plt.plot(storage_ts['timestamp'], storage_ts['Depression [mm]'])
plt.plot(storage_ts['timestamp'], storage_ts['Ponded Water [mm]'])
plt.show()

In [ ]:
plt.figure()
plt.plot(storage_ts['timestamp'], storage_ts['Soil Water[0] [mm]'])
plt.plot(storage_ts['timestamp'], storage_ts[' Cum. Inputs [mm]'])
plt.show()

In [ ]:
plt.figure()
plt.plot(storage_ts['timestamp'], storage_ts['Rivulet Storage [mm]'])
plt.plot(storage_ts['timestamp'], storage_ts['Channel Storage [mm]'])
plt.show()

### Single Season Storage

In [ ]:
years = 25
offset = (31 + 28) * 24 # all hours for Jan and Feb
spring_start = storage_ts.timestamp.iloc[0] + pd.Timedelta(hours=offset + int(years * 365.25 * 24))
spring_period = (31 + 30 + 31) * 24 # all ours for Mar, Apr, May
spring_end = spring_start + pd.Timedelta(hours=spring_period)

storage_ts_season = storage_ts[(storage_ts.timestamp >= spring_start) & (storage_ts.timestamp < spring_end)]


In [ ]:
plt.figure()
plt.plot(storage_ts_season['timestamp'], storage_ts_season['Depression [mm]'])
plt.plot(storage_ts_season['timestamp'], storage_ts_season['rainfall [mm/day]'])
plt.ylabel('Storage (mm)')
plt.title('Single Season Storage (Mar, Apr, May) \n Depressions (blue) and Rainfall (orange, mm/day)')
plt.show()

In [ ]:
# plt.plot(storage_ts_season['timestamp'], storage_ts_season['Soil Water[0] [mm]'])
# plt.plot(storage_ts_season['timestamp'], storage_ts_season[' Cum. Inputs [mm]'])
# plt.ylabel('Storage (mm)')
# plt.title('Single Season Storage (Mar, Apr, May) \n Soil Water (blue) and Cumulative Inputs (orange)')
# plt.show()

In [ ]:
plt.figure()
plt.plot(storage_ts_season['timestamp'], storage_ts_season['Rivulet Storage [mm]'])
plt.plot(storage_ts_season['timestamp'], storage_ts_season['Channel Storage [mm]'])
plt.ylabel('Storage (mm)')
plt.title('Single Season Storage (Mar, Apr, May) \n Rivulet (blue) and Channel (orange)')
plt.show()

## Testing Spotpy Setup

## Raven timeseries & Calibration results (Calibrate.ipynb)

In [ ]:
test = rv.RavenHydrograph(diagnostic_path, _run_id=run_id)
sim = test.read_sim(subbasin_id = "sub310019543", start_yr=2003, end_yr=2018, spring_only = False, refresh=True).reset_index(drop=True)
eval = test.read_obs(start_yr=2003, end_yr=2018, spring_only = False, refresh=True).reset_index(drop=True)
runoff = test.read_runoff(start_yr=2003, end_yr=2018, spring_only = False, refresh=True).reset_index(drop=True)

In [ ]:
rv.plot_timeseries(sim, eval, runoff, start_idx=0, end_idx=67000, timestep= 'H')


In [ ]:
rv.plot_timeseries(sim, eval, runoff, start_idx=67001, end_idx=len(sim), timestep = 'H')

### Single season hydrograph

In [ ]:

years = 6
offset = (31 + 28) * 24 # all hours for Jan and Feb
spring_start = int(offset + years * 365.25 * 24)
spring_period = (31 + 30 + 31) * 24 # all ours for Mar, Apr, May
spring_end = spring_start + spring_period

# For CWGC simulation on 3h timestep
# spring_start = int(spring_start / 3)
# spring_end = int(spring_end / 3)

rv.plot_timeseries(sim, eval, runoff, start_idx= spring_start, end_idx=spring_end, timestep= 'H')

In [ ]:
def index_to_date(index, start_date="2003-01-01 00:00:00", freq="H"):
    """
    Convert a zero-based index to a datetime object based on a given start date and frequency.
    """
    start = pd.to_datetime(start_date)
    return start + pd.to_timedelta(index, unit=freq[0].lower())

In [ ]:
index_to_date(220715, start_date="1980-01-01 00:00:00", freq = "H")

# Timeseries By HRU

## Reading storage timeseries by HRU

In [ ]:
# Required user specifications (specified above, uncomment as needed)
# model_str = "Cypress_New"


In [ ]:
def clean_hru_dataframe(_model_name: str = 'Cypress_New' , 
                        _param_name: str = '',
                        _param_type: str = '_Continuous_Maximum',
                        _model_path: str = '') -> pd.DataFrame:
    """ 
    Cleans and processes the HRU dataframe read from a CSV file.

    csv includes unnecessary columns and a unit row, leading to a messy dataframe.
    """
    # Load csv
    file_name = _model_name + '_' + _param_name + _param_type + '_ByHRU.csv'
    file_path = _model_path + file_name

    hru_df = pd.read_csv(file_path, low_memory=False)

    # rename columns and drop units from 1st row
    hru_df.columns = ['time_hrs', 'day', 'hour'] + hru_df.columns.tolist()[3:] 
    hru_df.drop(0, inplace=True)
    hru_df.reset_index(drop=True, inplace=True)

    # Calculate a single datetime columnm drop day and hour columns, and move timestamp to the front
    hru_df['timestamp'] = pd.to_datetime(hru_df['day'] + " " + hru_df['hour'].str[:8], format='%Y-%m-%d %H:%M:%S')
    hru_df.drop(['day', 'hour'], axis=1, inplace=True)
    hru_df['time_hrs'] = (hru_df['time_hrs'].astype(float) * 24).round().astype(int)  # convert days to hours
    cols = ['timestamp'] +[col for col in hru_df.columns if (col != 'timestamp') & ('Unnamed' not in col)] # Also drop any additional unnamed columns
    hru_df = hru_df[cols] 

    # Convert all data columns to numeric (float)
    cols = hru_df.columns[2:].tolist()
    hru_df[cols] = hru_df[cols].astype(float)

    return hru_df


In [ ]:
hru_dep_ts = clean_hru_dataframe(_param_name='DEPRESSION', _model_path=diagnostic_path + f'output{test.run_id}/')
hru_rr_ts = clean_hru_dataframe(_param_name='FROM_ATMOS_PRECIP', _model_path=diagnostic_path + f'output{test.run_id}/')
hru_seep_ts = clean_hru_dataframe(_param_name='TO_SOIL[0]', _model_path=diagnostic_path + f'output{test.run_id}/')
hru_catch_ts = clean_hru_dataframe(_param_name='TO_SURFACE_WATER', _model_path=diagnostic_path + f'output{test.run_id}/')

## Reading outflow timeseries by HRU

In [ ]:
disch_ts = pd.read_csv(diagnostic_path + f'/output{test.run_id}/Cypress_New_Hydrographs.csv')

In [ ]:
def clean_disch_dataframe(_model_name: str = 'Cypress_New',
                        _model_path: str = '') -> pd.DataFrame:
    """ 
    Cleans and processes the HRU dataframe read from a CSV file.

    csv includes unnecessary columns and a unit row, leading to a messy dataframe.
    """
    # Load csv
    file_name = _model_name + '_Hydrographs.csv'
    file_path = _model_path + file_name

    disch_df = pd.read_csv(file_path, low_memory=False)

    # Calculate a single datetime columnm drop day and hour columns, and move timestamp to the front
    disch_df['timestamp'] = pd.to_datetime(disch_df['date'] + " " + disch_df['hour'].str[:8], format='%Y-%m-%d %H:%M:%S')
    disch_df.drop(['date', 'hour'], axis=1, inplace=True)
    disch_df['time_hrs'] = (disch_df['time'].astype(float) * 24).round().astype(int)  # convert days to hours
    disch_df.drop('time', axis=1, inplace=True)
    cols = ['timestamp'] +[col for col in disch_df.columns if (col != 'timestamp') & ('Unnamed' not in col)] # Also drop any additional unnamed columns
    disch_df = disch_df[cols]

    return disch_df

In [ ]:
disch_ts = clean_disch_dataframe(_model_name='Cypress_New', _model_path=diagnostic_path + f'output{test.run_id}/')

## Converting cumulative data to continuous / incremental

In [ ]:
def cumsum_to_incremental(cum_df: pd.Series, _invert: bool = False) -> pd.Series:
    """
    Convert a cumulative sum series to an incremental series.
    
    Parameters:
    test_col (pd.Series): A pandas Series representing cumulative sums.
    invert (bool): If True, the incremental values are inverted (multiplied by -1).
    
    Returns:
    pd.Series: A pandas Series representing incremental values.
    """

    def columnwise_fn(test_col: pd.Series,
                            _invert_: bool = False) -> pd.Series:

        out_col = pd.Series(index=test_col.index, dtype=float)
        
        out_col = test_col.diff().fillna(test_col.iloc[0])
        # for i in range(len(test_col)):
        #     if i == 0:
        #         out_col[i] = test_col[i]
        #     else:
        #         out_col[i] = test_col[i] - test_col[i-1]

        if _invert_:
            out_col = out_col * -1
        
        # print('Normalized total from original cumsum:' + str(test_col.iat[-1]/len(test_col))+ ' new average value:' + str(out_col.mean()))
        
        return out_col
    
    inc_df = cum_df.apply(lambda col: columnwise_fn(col, _invert_=_invert) if '310' in col.name else col)

    return inc_df


Note: Generating the incremental timeseries takes time. So these are stored in .pkl files instead. FOr a new raven simulation, uncomment the necessary code.

In [ ]:
## Long run time, only uncomment when needed

hru_rr_inc = cumsum_to_incremental(hru_rr_ts, _invert=True)
hru_seep_inc = cumsum_to_incremental(hru_seep_ts)
hru_catch_inc = cumsum_to_incremental(hru_catch_ts)

import pickle

# with open(DATA_CAL + 'output/hru_rr_inc.pkl', 'wb') as f:
#     pickle.dump(hru_rr_inc, f)

# with open(DATA_CAL + 'output/hru_seep_inc.pkl', 'wb') as f:
#     pickle.dump(hru_seep_inc, f)

# with open(DATA_CAL + 'output/hru_catch_inc.pkl', 'wb') as f:
#     pickle.dump(hru_catch_inc, f)

# with open(DATA_CAL + 'output/hru_rr_inc.pkl', 'rb') as f:
#     hru_rr_inc = pickle.load(f)

# with open(DATA_CAL + 'output/hru_seep_inc.pkl', 'rb') as f:
#     hru_seep_inc = pickle.load(f)

# with open(DATA_CAL + 'output/hru_catch_inc.pkl', 'rb') as f:
#     hru_catch_inc = pickle.load(f)

## Plot storage and flows

Plotting functions

In [ ]:
# Convert mm/h to mm/day
cols = hru_rr_inc.columns[2:].tolist()
hru_rr_inc[cols] = hru_rr_inc[cols] * 24

In [ ]:



def plot_hru_ts(_hru_ts: pd.DataFrame, _hru_id: str, dataset: str, _start_yr: int = 1999, _end_yr: int = 2016):
    start_date = f"{_start_yr}-01-01 00:00:00"
    end_date = f"{_end_yr}-12-31 23:00:00"

    clipped_ts = _hru_ts[(_hru_ts['timestamp'] >= pd.to_datetime(start_date)) & (_hru_ts['timestamp'] <= pd.to_datetime(end_date))]
    plt.figure(figsize=(15, 5))
    plt.plot(clipped_ts['timestamp'], clipped_ts[_hru_id], label=dataset)
    plt.title(dataset)
    plt.show()

def plot_hru_disch(_disch_ts: pd.DataFrame, _hru_id: str, dataset: str, _start_yr: int = 1999, _end_yr: int = 2016):
    start_date = f"{_start_yr}-01-01 00:00:00"
    end_date = f"{_end_yr}-12-31 23:00:00"

    clipped_ts = _disch_ts[(_disch_ts['timestamp'] >= pd.to_datetime(start_date)) & (_disch_ts['timestamp'] <= pd.to_datetime(end_date))]
    col_name = clipped_ts.columns[clipped_ts.columns.str.contains(_hru_id)].tolist()

    plt.figure(figsize=(15, 5))
    plt.plot(clipped_ts['timestamp'], clipped_ts[col_name], label=dataset)
    plt.title(dataset)
    plt.show()

Plotting a full timeseries

In [ ]:

hru_id = '310020581'
start_yr = 1999
end_yr = 2016

plot_hru_ts(hru_rr_inc, hru_id, 'Runoff (PHyDAP)')
plot_hru_ts(hru_dep_ts, hru_id, 'Depression Storage')
plot_hru_ts(hru_seep_inc, hru_id, 'Seepage to Soil')
plot_hru_ts(hru_catch_inc, hru_id, 'Catchment Routing')
plot_hru_disch(disch_ts, hru_id, 'Discharge at Outlet (m3/s)')


Plotting a single season

In [ ]:
hru_id = '310020581'
single_year = 2009

plot_hru_ts(hru_dep_ts, hru_id, 'Depression Storage', _start_yr=start_yr, _end_yr=end_yr)
plot_hru_ts(hru_rr_inc, hru_id, 'Runoff (PHyDAP)', _start_yr=single_year, _end_yr=single_year)
plot_hru_ts(hru_seep_inc, hru_id, 'Seepage to Soil', _start_yr=start_yr, _end_yr=end_yr)
plot_hru_ts(hru_catch_inc, hru_id, 'Catchment Routing', _start_yr=start_yr, _end_yr=end_yr)
plot_hru_disch(disch_ts, hru_id, 'Discharge at Outlet (m3/s)', _start_yr=start_yr, _end_yr=end_yr)